# Agentic Retrieval-Augmented Generation (RAG) System

This notebook builds a document question-answering system using a RAG architecture.

Pipeline:

PDF → Chunking → Embeddings → Vector Database → Semantic Retrieval → LLM Answer

Technologies used:

- LangChain
- Sentence Transformers (all-MiniLM-L6-v2)
- Qdrant Vector Database
- Llama 3.1 via Groq
- Gradio Web Interface

# Imports

In [1]:
import os
import numpy as np
from pypdf import PdfReader

from dotenv import load_dotenv

from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.embeddings import HuggingFaceEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

from langchain_groq import ChatGroq

import gradio as gr


# load environment variables
load_dotenv()

# read API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Loaded:", GROQ_API_KEY is not None)

API Loaded: True


# Extracting Text from PDFs

In [2]:
reader = PdfReader("PritamMohanta_Resume.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:500])

Pritam Mohanta
pritamohanta.in@gmail.com | +917478449766 | LinkedIn | GitHub | Portfolio
Education
Jalpaiguri Government Engineering College Sept 2023 – Expected Jun 2027
• Bachelor of Technology in Information Technology CGPA: 8.42/10.0
• Relevant Coursework: Data Structures & Algorithms, Operating Systems, DBMS, Computer Networks, OOP
Projects
CF Performance Mirror — Browser Extension GitHub | Chrome | Firefox
JavaScript, Web Extensions API, DOM Manipulation, REST APIs
• Built a fully client-s


In [3]:
from langchain.schema import Document

documents = [Document(page_content=text)]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 10


In [5]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
   model_name="BAAI/bge-base-en-v1.5",
)

embeddings = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Number of embeddings:", len(embeddings))
print("Embedding vector size:", len(embeddings[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of embeddings: 10
Embedding vector size: 768


In [6]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")
collection_name = "rag_collection"

In [7]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

True

In [8]:
points = []

for i, embedding in enumerate(embeddings):
    points.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": chunks[i].page_content
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print("Embeddings stored!")

Embeddings stored!


In [9]:
@tool
def search_resume(question: str):
    """
    Search the resume and return relevant sections.

    Use this tool whenever a question asks about:
    - projects
    - skills
    - education
    - experience
    - achievements
    """

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=5
    )

    print("\nRetrieved Chunks:\n")

    for r in results.points:
        print(r.payload["text"])
        print("\n----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points])

    return context

In [10]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key= GROQ_API_KEY,
    temperature=0
)

In [11]:
from langchain.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""
You are an intelligent assistant that analyzes resumes.

You have access to the following tools:

{tools}

When answering a question:
1. Always use the search_resume tool first.
2. Read the retrieved context carefully.
3. Answer the question using only the retrieved information.

Use the following format:

Question: the input question
Thought: you should think about what to do
Action: the tool to use, should be one of [{tool_names}]
Action Input: the input to the tool
Observation: the result of the tool
... (this Thought/Action/Observation can repeat)
Thought: I now know the final answer
Final Answer: the answer to the question

Question: {input}

{agent_scratchpad}
""")

In [12]:
tools = [search_resume]

agent = create_react_agent(
    llm,
    tools,
    react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [13]:
def ask_question(question):
    try:
        response = agent_executor.invoke(
            {"input": question}
        )

        return response.get("output", str(response))

    except Exception as e:
        return str(e)

In [14]:
def ask_llm(context, question):
    prompt = f"""
You are an AI assistant that answers questions based only on the provided context.

Context:
{context}

Question:
{question}

Answer clearly and concisely using only the context.
"""

    response = llm.invoke(prompt)

    return response.content

In [15]:
def rag_query(question):

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=5
    )

    print("\nRetrieved Chunks:\n")

    for r in results.points:
        print(r.payload["text"])
        print("\n-----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points])

    answer = ask_llm(context, question)

    return answer

In [16]:
print(rag_query("What is this document about?"))


Retrieved Chunks:

access control, and secure session management for recruiters and job seekers.
• Implemented scalable file upload pipeline using Multer and Cloudinary for resume storage and profile images;
designed RESTful API endpoints with MongoDB for efficient data persistence and retrieval.
Achievements & A wards
• ACM-ICPC Kanpur Regional 2025: Secured Rank 57 in onsite round.

-----------------

(1850+)
• Solved 3000+ problems across Codeforces, CodeChef, LeetCode, SPOJ, and AtCoder.
Technical Skills
Languages: C, C++, Java, JavaScript
Web Technologies: React.js, Node.js, Express.js, RESTful APIs, Tailwind CSS
Databases & Tools: MongoDB, MySQL, Git, GitHub, Linux, Vercel, Postman
Core Competencies: Data Structures & Algorithms, Operating System, Object-Oriented Programming, Computer

-----------------

Projects
CF Performance Mirror — Browser Extension GitHub | Chrome | Firefox
JavaScript, Web Extensions API, DOM Manipulation, REST APIs
• Built a fully client-side cross-browse

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [18]:
import gradio as gr

def chat_interface(question):
    return rag_query(question)

interface = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="Agentic RAG with Llama3",
    description="Ask questions about your document"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



Retrieved Chunks:

access control, and secure session management for recruiters and job seekers.
• Implemented scalable file upload pipeline using Multer and Cloudinary for resume storage and profile images;
designed RESTful API endpoints with MongoDB for efficient data persistence and retrieval.
Achievements & A wards
• ACM-ICPC Kanpur Regional 2025: Secured Rank 57 in onsite round.

-----------------

• Scaled to 200+ installs and 100+ DAUs across 15+ countries; implemented MutationObserver-driven
adaptive theming, rating-aware division classification, and persistent cross-session settings.
Psh — POSIX-Compatible Unix Shell in C GitHub
C, Linux System Calls, Process Management, POSIX APIs, Memory Management

-----------------

Projects
CF Performance Mirror — Browser Extension GitHub | Chrome | Firefox
JavaScript, Web Extensions API, DOM Manipulation, REST APIs
• Built a fully client-side cross-browser (Chrome/Firefox) extension analyzing 10K+ submissions via Codeforces
REST API, de

Traceback (most recent call last):
  File "/home/perxeuss/Desktop/agentic-rag-llama3/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/perxeuss/Desktop/agentic-rag-llama3/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/perxeuss/Desktop/agentic-rag-llama3/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/perxeuss/Desktop/agentic-rag-llama3/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/perxeuss